In [1]:
import datetime as dt
import os
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

import concurrent.futures
import itertools
from multiprocessing import Pool, cpu_count

import dask.array as da
import dask.dataframe as dd
import pandas as pd
import numpy as np
import statsmodels.api as sm
from dotenv import load_dotenv
from mc_postgres_db import models as mc
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, aliased
from statsmodels.regression.rolling import RollingOLS
from statsmodels.tsa.stattools import adfuller
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from utils.data import align_series, freq_to_window

## Backtesting Cointegration on N Currencies

In this notebook I will apply my pairs trading strategy against historical data. I will attempt to create this using vectorized functions so that I can iterate quickly against large amounts of historical data (around 1 year or longer). The implementation leverages NumPy's vectorized operations to achieve computational efficiency, enabling rapid backtesting across extended temporal horizons while maintaining statistical rigor. This approach is particularly crucial for cointegration analysis, where the computational complexity of traditional iterative methods scales exponentially with the number of currency pairs and observation periods. By employing vectorized computations, we can efficiently evaluate the statistical significance of cointegration relationships across multiple timeframes and conduct comprehensive robustness checks that are essential for validating the empirical validity of our trading strategy in diverse market conditions.

I will start by setting up my environment to pull historical data.

In [2]:
load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

Now, I will define the parameters of my test in terms of the period I would like to iterate over and anything else that is important to the simulation.

In [3]:
start = dt.datetime(2024, 1, 1)
end = dt.datetime(2024, 1, 3)

After our parameters are defined, we can pull our historical data.

In [4]:
# Get the Kraken provider.
with Session(engine) as session:
    stmt = select(mc.Provider).where(mc.Provider.name == "Kraken")
    kraken = session.execute(stmt).scalar_one()

# Get the USD asset.
with Session(engine) as session:
    stmt = select(mc.Asset).where(mc.Asset.name == "USD")
    usd = session.execute(stmt).scalar_one()

# Get all asset pairs from Kraken.
from_asset = aliased(mc.Asset)
to_asset = aliased(mc.Asset)
data = pd.read_sql(
    select(
        mc.ProviderAssetMarket.timestamp,
        mc.ProviderAssetMarket.from_asset_id,
        mc.ProviderAssetMarket.to_asset_id,
        from_asset.name.label("from_asset_name"),
        to_asset.name.label("to_asset_name"),
        mc.ProviderAssetMarket.close,
    )
    .join(to_asset, mc.ProviderAssetMarket.to_asset_id == to_asset.id)
    .join(from_asset, mc.ProviderAssetMarket.from_asset_id == from_asset.id)
    .where(
        mc.ProviderAssetMarket.provider_id == kraken.id,
        mc.ProviderAssetMarket.from_asset_id == usd.id,
        mc.ProviderAssetMarket.timestamp >= start,
        mc.ProviderAssetMarket.timestamp <= end,
    )
    .order_by(mc.ProviderAssetMarket.timestamp),
    engine,
)
display(data)

,timestamp,from_asset_id,to_asset_id,from_asset_name,to_asset_name,close
0,2024-01-01,2,1,USD,BTC,42260.30000
1,2024-01-01,2,3,USD,ETH,2281.76000
2,2024-01-01,2,4,USD,XRP,0.61500
3,2024-01-01,2,7,USD,SOL,101.76000
4,2024-01-01,2,8,USD,AAVE,108.68000
...,...,...,...,...,...,...
45912,2024-01-03,2,43,USD,ALGO,0.22710
45913,2024-01-03,2,45,USD,ATOM,10.96900
45914,2024-01-03,2,52,USD,FIL,7.17800
45915,2024-01-03,2,53,USD,FLR,0.01876


From these currencies, we then want to find every possible pair combination so that we can check for cointegration. I will first find these combinations and then generate a new data frame that we can perform cointegration on at once.

In [ ]:
# Get the combinations of currencies.
timeframe = pd.date_range(start=start, end=end, freq="1min")
currency_ids = (
    data[["from_asset_id", "to_asset_id"]]
    .drop_duplicates()
    .apply(tuple, axis=1)
    .tolist()
)
currency_id_combinations = list(itertools.combinations(currency_ids, 2))
print(
    f"There are {len(currency_id_combinations)} combinations of currencies in the data."
)


# For very large datasets, you can also use Dask for the cartesian product
def create_calculation_frame_dask(timeframe, currency_combinations):
    """Create calculation frame using Dask for memory efficiency"""

    def process_chunk(chunk_combinations):
        chunk_data = [
            {
                "timestamp": timestamp,
                "from_asset_id_1": from_asset_id[0],
                "to_asset_id_1": from_asset_id[1],
                "from_asset_id_2": to_asset_id[0],
                "to_asset_id_2": to_asset_id[1],
            }
            for timestamp in timeframe
            for from_asset_id, to_asset_id in chunk_combinations
        ]
        chunk_data = pd.DataFrame(chunk_data)
        chunk_data = chunk_data.merge(
            data.rename(
                columns={
                    "from_asset_id": "from_asset_id_1",
                    "to_asset_id": "to_asset_id_1",
                    "from_asset_name": "from_asset_name_1",
                    "to_asset_name": "to_asset_name_1",
                    "close": "close_1",
                }
            ),
            on=["timestamp", "from_asset_id_1", "to_asset_id_1"],
            how="left",
        )
        chunk_data = chunk_data.merge(
            data.rename(
                columns={
                    "from_asset_id": "from_asset_id_2",
                    "to_asset_id": "to_asset_id_2",
                    "from_asset_name": "from_asset_name_2",
                    "to_asset_name": "to_asset_name_2",
                    "close": "close_2",
                }
            ),
            on=["timestamp", "from_asset_id_2", "to_asset_id_2"],
            how="left",
        )
        chunk_data[
            [
                "close_1",
                "close_2",
                "from_asset_name_1",
                "to_asset_name_1",
                "from_asset_name_2",
                "to_asset_name_2",
            ]
        ] = (
            chunk_data.sort_values(by="timestamp")
            .groupby(
                [
                    "from_asset_id_1",
                    "to_asset_id_1",
                    "from_asset_id_2",
                    "to_asset_id_2",
                ],
            )[
                [
                    "close_1",
                    "close_2",
                    "from_asset_name_1",
                    "to_asset_name_1",
                    "from_asset_name_2",
                    "to_asset_name_2",
                ]
            ]
            .ffill()
        )
        chunk_data.dropna(inplace=True)
        return chunk_data

    # Split the calculation frame into chunks to process in parallel.
    chunks = []
    chunk_size = 10
    chunk_indices = list(range(0, len(currency_combinations), chunk_size))
    chunk_combinations_list = [
        currency_combinations[i : i + chunk_size] for i in chunk_indices
    ]
    with concurrent.futures.ThreadPoolExecutor() as executor:
        results = list(
            tqdm(
                executor.map(process_chunk, chunk_combinations_list),
                total=len(chunk_combinations_list),
            )
        )
        chunks.extend(results)

    # Combine all chunks into a single Dask DataFrame
    dask_chunks = [dd.from_pandas(chunk, npartitions=2) for chunk in chunks]

    # Set the optimal partition size for your use case.
    calculation_frame = dd.concat(dask_chunks)
    calculation_frame = calculation_frame.repartition(partition_size="100MB")

    return calculation_frame


# Use the Dask-optimized function
calculation_frame = create_calculation_frame_dask(timeframe, currency_id_combinations)
print(f"The calculation frame has {len(calculation_frame)} rows.")

# Get the first 1000 rows of the calculation frame.
calculation_frame.head(1000)

There are 496 combinations of currencies in the data.


  0%|          | 0/50 [00:00<?, ?it/s]

The calculation frame has 1410317 rows.


,timestamp,from_asset_id_1,to_asset_id_1,from_asset_id_2,to_asset_id_2,from_asset_name_1,to_asset_name_1,close_1,from_asset_name_2,to_asset_name_2,close_2
0,2024-01-01 00:00:00,2,1,2,3,USD,BTC,42260.3,USD,ETH,2281.760000
1,2024-01-01 00:00:00,2,1,2,4,USD,BTC,42260.3,USD,XRP,0.615000
2,2024-01-01 00:00:00,2,1,2,7,USD,BTC,42260.3,USD,SOL,101.760000
3,2024-01-01 00:00:00,2,1,2,8,USD,BTC,42260.3,USD,AAVE,108.680000
4,2024-01-01 00:00:00,2,1,2,9,USD,BTC,42260.3,USD,XLM,0.128940
...,...,...,...,...,...,...,...,...,...,...,...
995,2024-01-01 01:39:00,2,1,2,10,USD,BTC,42720.5,USD,DOGE,0.090107
996,2024-01-01 01:39:00,2,1,2,12,USD,BTC,42720.5,USD,ADA,0.602292
997,2024-01-01 01:39:00,2,1,2,20,USD,BTC,42720.5,USD,BCH,259.810000
998,2024-01-01 01:39:00,2,1,2,21,USD,BTC,42720.5,USD,AVAX,39.620000


/Users/glynfinck/Documents/Software Projects/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/dask/dataframe/groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
/Users/glynfinck/Documents/Software Projects/Repositories/mc-notebooks/.venv/lib/python3.12/site-packages/dask/dataframe/groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(f

Now that we have a calculation frame, we can start computing our rolling cointegration which will be starting point for our algorithm. This is because for our algorithm to work the pair must be cointegrated over some rolling period previous to the current timestamp.

For cointegration, we will perform a rolling regression of the following equation:

$$
X_t^1 = \alpha + \beta X_t^2 + \epsilon_t
$$

Where $X_t^1$ is the price of the first currency, $X_t^2$ is the price of the second currency, $\alpha$ is the y-intercept of the linear regression, $\beta$ is the slope, and $\epsilon_t$ is the time dependant error of the regression. For our cointegration test, we will test if $\epsilon_t$ is stationary. For calculating stationarity of $\epsilon_t$, we will use the Augmented Dickey-Fuller (ADF) unit root test on the residuals. For now, I will use the function `adfuller` from `statsmodels`.

First we will compute the rolling OLS of each pair combination.

In [27]:
ROLLING_OLS_COLUMNS = {
    "timestamp": "datetime64[ns]",
    "alpha": float,
    "beta": float,
    "p_value": float,
}


# Define the rolling function that returns only p-value
def rolling_adf_p_value(
    series_1: np.ndarray, series_2: np.ndarray, alpha: float, beta: float
) -> np.float64:
    """Calculate ADF p-value for a rolling window"""
    if len(series_1) != len(series_2):
        raise ValueError(
            "The parameters series_1 and series_2 must be the same length."
        )
    if len(series_1) < 2:
        return np.nan
    try:
        # Return only the p-value (index 1)
        return adfuller(series_2 - alpha - beta * series_1)[1]
    except:
        return np.nan


def rolling_ols(df: pd.DataFrame, window: int = 100, step: int = 1) -> pd.DataFrame:
    # If the window is larger than the number of rows, return a DataFrame with None values.
    if len(df) < window:
        return pd.DataFrame(
            {
                "timestamp": df["timestamp"],
                "alpha": [np.nan] * len(df),
                "beta": [np.nan] * len(df),
                "p_value": [np.nan] * len(df),
            }
        )

    # Compute the rolling OLS.
    X = df["close_1"]
    y = df["close_2"]
    X = sm.add_constant(X)
    result = (
        RollingOLS(y, X, window=window)
        .fit()
        .params.rename(columns={"const": "alpha", "close_1": "beta"})
    )
    result["timestamp"] = df["timestamp"]
    result["close_1"] = df["close_1"]
    result["close_2"] = df["close_2"]

    # Compute the ADF unit root test on the residuals using a for loop.
    p_values = np.zeros(len(result)) * np.nan
    alpha_arr = result["alpha"].values
    beta_arr = result["beta"].values
    close_1_arr = result["close_1"].values
    close_2_arr = result["close_2"].values
    for i in range(window - 1, len(result), step):
        window_alpha = alpha_arr[i]
        window_beta = beta_arr[i]
        window_close_1 = close_1_arr[i - window + 1 : i + 1]
        window_close_2 = close_2_arr[i - window + 1 : i + 1]
        p_val = rolling_adf_p_value(
            window_close_1, window_close_2, window_alpha, window_beta
        )
        p_values[i] = p_val
    result["p_value"] = p_values
    result["p_value"] = result["p_value"].ffill()

    # Only output the same columns as the input.
    output = result[list(ROLLING_OLS_COLUMNS.keys())]

    return output.reset_index(drop=True)

In [30]:
groupby_result = (
    calculation_frame.groupby(
        [
            "from_asset_id_1",
            "to_asset_id_1",
            "from_asset_id_2",
            "to_asset_id_2",
        ]
    )
    .apply(
        lambda x: rolling_ols(
            x.reset_index(drop=True),
            window=freq_to_window("1m", "1D"),
            step=freq_to_window("1m", "1D"),
        ),
        meta=ROLLING_OLS_COLUMNS,
    )
    .reset_index()
    .compute()
)

calculation_frame_finished = calculation_frame.merge(
    groupby_result,
    on=[
        "timestamp",
        "from_asset_id_1",
        "to_asset_id_1",
        "from_asset_id_2",
        "to_asset_id_2",
    ],
    how="left",
)

In [46]:
calculation_frame_finished["spread"] = (
    calculation_frame_finished["close_1"]
    - calculation_frame_finished["alpha"]
    - calculation_frame_finished["beta"] * calculation_frame_finished["close_2"]
)
calculation_frame_finished["spread_mean"] = calculation_frame_finished.groupby(
    ["from_asset_id_1", "to_asset_id_1", "from_asset_id_2", "to_asset_id_2"]
).apply(lambda x: x["spread"].rolling(window=freq_to_window("1m", "1D")).mean())
calculation_frame_finished["spread_std"] = calculation_frame_finished.groupby(
    ["from_asset_id_1", "to_asset_id_1", "from_asset_id_2", "to_asset_id_2"]
).apply(lambda x: x["spread"].rolling(window=freq_to_window("1m", "1D")).std())
calculation_frame_finished["z_score"] = (
    calculation_frame_finished["spread"] - calculation_frame_finished["spread_mean"]
) / calculation_frame_finished["spread_std"]
calculation_frame_finished.compute()

/var/folders/tc/qpkxsjrx4s72nqyhf67_1ctw0000gn/T/ipykernel_90934/3904285145.py:2: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  calculation_frame_finished["spread_mean"] = calculation_frame_finished.groupby(["from_asset_id_1", "to_asset_id_1", "from_asset_id_2", "to_asset_id_2"]).apply(lambda x: x["spread"].rolling(window=freq_to_window("1m", "1D")).mean())


TypeError: Column assignment doesn't support type <class 'dask.dataframe.dask_expr._collection.DataFrame'>

In [38]:
calculation_frame_trade = calculation_frame_finished.loc[
    calculation_frame_finished["p_value"] < 0.01
].compute()
calculation_frame_trade = calculation_frame_trade.reset_index(drop=True)
calculation_frame_trade

,timestamp,from_asset_id_1,to_asset_id_1,from_asset_id_2,to_asset_id_2,from_asset_name_1,to_asset_name_1,close_1,from_asset_name_2,to_asset_name_2,close_2,level_4,alpha,beta,p_value
0,2024-01-01 23:59:00,2,1,2,12,USD,BTC,44180.80000,USD,ADA,0.622753,1439,-0.120118,1.692032e-05,5.272083e-03
1,2024-01-02 00:00:00,2,1,2,12,USD,BTC,44164.50000,USD,ADA,0.623017,1440,-0.119266,1.690038e-05,5.272083e-03
2,2024-01-02 00:01:00,2,1,2,12,USD,BTC,44234.30000,USD,ADA,0.622905,1441,-0.118156,1.687436e-05,5.272083e-03
3,2024-01-02 00:02:00,2,1,2,12,USD,BTC,44210.80000,USD,ADA,0.623431,1442,-0.117224,1.685255e-05,5.272083e-03
4,2024-01-02 00:03:00,2,1,2,12,USD,BTC,44211.00000,USD,ADA,0.623301,1443,-0.116296,1.683081e-05,5.272083e-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187107,2024-01-03 00:00:00,2,54,2,28,USD,FET,0.73690,USD,DAI,0.999970,2845,0.998162,1.789234e-03,4.272696e-07
187108,2024-01-03 00:00:00,2,54,2,41,USD,FET,0.73690,USD,POL,0.997130,2354,0.997130,6.366463e-12,1.483622e-05
187109,2024-01-03 00:00:00,2,14,2,28,USD,SUI,0.90730,USD,DAI,0.999970,2845,0.993696,6.544667e-03,2.928563e-07
187110,2024-01-03 00:00:00,2,14,2,41,USD,SUI,0.90730,USD,POL,0.997130,2354,0.997130,1.591616e-12,5.681053e-06


In [41]:
calculation_frame_trade["spread"] = (
    calculation_frame_trade["close_1"]
    - calculation_frame_trade["alpha"]
    - calculation_frame_trade["beta"] * calculation_frame_trade["close_2"]
)
calculation_frame_trade

,timestamp,from_asset_id_1,to_asset_id_1,from_asset_id_2,to_asset_id_2,from_asset_name_1,to_asset_name_1,close_1,from_asset_name_2,to_asset_name_2,close_2,level_4,alpha,beta,p_value,spread
0,2024-01-01 23:59:00,2,1,2,12,USD,BTC,44180.80000,USD,ADA,0.622753,1439,-0.120118,1.692032e-05,5.272083e-03,44180.920107
1,2024-01-02 00:00:00,2,1,2,12,USD,BTC,44164.50000,USD,ADA,0.623017,1440,-0.119266,1.690038e-05,5.272083e-03,44164.619256
2,2024-01-02 00:01:00,2,1,2,12,USD,BTC,44234.30000,USD,ADA,0.622905,1441,-0.118156,1.687436e-05,5.272083e-03,44234.418146
3,2024-01-02 00:02:00,2,1,2,12,USD,BTC,44210.80000,USD,ADA,0.623431,1442,-0.117224,1.685255e-05,5.272083e-03,44210.917214
4,2024-01-02 00:03:00,2,1,2,12,USD,BTC,44211.00000,USD,ADA,0.623301,1443,-0.116296,1.683081e-05,5.272083e-03,44211.116286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187107,2024-01-03 00:00:00,2,54,2,28,USD,FET,0.73690,USD,DAI,0.999970,2845,0.998162,1.789234e-03,4.272696e-07,-0.263051
187108,2024-01-03 00:00:00,2,54,2,41,USD,FET,0.73690,USD,POL,0.997130,2354,0.997130,6.366463e-12,1.483622e-05,-0.260230
187109,2024-01-03 00:00:00,2,14,2,28,USD,SUI,0.90730,USD,DAI,0.999970,2845,0.993696,6.544667e-03,2.928563e-07,-0.092940
187110,2024-01-03 00:00:00,2,14,2,41,USD,SUI,0.90730,USD,POL,0.997130,2354,0.997130,1.591616e-12,5.681053e-06,-0.089830
